# 临时脚本

In [ ]:
!cd ../cpp && make clean && make && cd ..

In [ ]:
from tvm_book.tvm_ext.libinfo import load_lib

_LIB_EXT, _LIB_EXT_NAME = load_lib(name="libtvm_ext.so", search_path=["../cpp/outputs/libs"])

In [ ]:
import tvm_book.tvm_ext.testing._ffi_api as ffi_api

In [ ]:
import asyncio
from tqdm.asyncio import tqdm

async def worker(id, is_open=1):
    await asyncio.sleep(2)
    # if is_open:
    #     # async with semaphore:
    #         # print(f"Task {id} acquiring semaphore")
        
    #         # print(f"Task {id} releasing semaphore")
    # else:
    #     print(f"Task {id} acquiring semaphore")
    #     await asyncio.sleep(2)
    #     print(f"Task {id} releasing semaphore")

semaphore = asyncio.Semaphore(20)  # 允许最多两个协程同时运行
semaphore
tasks = [asyncio.gather(*[asyncio.create_task(worker(i)) for i in range(20)]) for _ in range(10)]
async with asyncio.Semaphore(20):
    res = await tqdm.gather(*tasks)

100%|██████████| 10/10 [00:20<00:00,  2.00s/it]


In [26]:
import asyncio
import random
import time

async def task(val):
    return val * 2

async def worker(name, queue, res):
    while 1:
        # 从队列获取一个“工作项”。
        sleep_for = await queue.get()

        # 休眠 "sleep_for" 秒。
        res.append(await task(sleep_for))

        # 通知队列“工作项”已被处理。
        queue.task_done()

        print(f'{name} has slept for {sleep_for:.2f} seconds')


async def main():
    res = []
    # 创建一个用于存储我们的“工作项”的队列。
    queue = asyncio.Queue()

    # 生成随机时段并将它们放入队列。
    total_sleep_time = 0
    for _ in range(20):
        sleep_for = random.uniform(0.05, 1.0)
        total_sleep_time += sleep_for
        queue.put_nowait(sleep_for)

    # 创建三个工作任务来并发地处理队列。
    tasks = []
    for i in range(3):
        task = asyncio.create_task(worker(f'worker-{i}', queue, res))
        tasks.append(task)

    # 等待直到队列处理完毕。
    started_at = time.monotonic()
    await queue.join()
    total_slept_for = time.monotonic() - started_at

    # 取消我们的工作任务。
    for task in tasks:
        task.cancel()
    # 等待直到所有工作任务都被取消。
    await asyncio.gather(*tasks, return_exceptions=True)

    print('====')
    print(f'3 workers slept in parallel for {total_slept_for:.2f} seconds')
    print(f'total expected sleep time: {total_sleep_time:.2f} seconds')
    return res


ts = await main()

worker-0 has slept for 0.27 seconds
worker-0 has slept for 0.67 seconds
worker-0 has slept for 0.38 seconds
worker-0 has slept for 0.70 seconds
worker-0 has slept for 0.76 seconds
worker-0 has slept for 0.44 seconds
worker-0 has slept for 0.52 seconds
worker-0 has slept for 0.77 seconds
worker-0 has slept for 0.79 seconds
worker-0 has slept for 0.81 seconds
worker-0 has slept for 0.37 seconds
worker-0 has slept for 0.59 seconds
worker-0 has slept for 0.93 seconds
worker-0 has slept for 0.89 seconds
worker-0 has slept for 0.28 seconds
worker-0 has slept for 0.94 seconds
worker-0 has slept for 0.30 seconds
worker-0 has slept for 0.84 seconds
worker-0 has slept for 0.36 seconds
worker-0 has slept for 0.37 seconds
====
3 workers slept in parallel for 0.00 seconds
total expected sleep time: 11.98 seconds


In [2]:
import os
os.cpu_count()

48